# 01 · Summary statistics and Table 1

**Module:** 3 · **Time:** 45 minutes · **Prerequisites:** Module 2: pandas basics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/03_describe_visualize/01_summary_statistics.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F03_describe_visualize%2F01_summary_statistics.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** Badges point to `main`; the current repository is private and these lessons remain in a draft PR.

**Completed solutions edition.**

## Learning goals

- **C**ompute grouped and overall descriptive statistics.
- **C**onstruct a weighted mean and a cross-tabulation.
- **E**xport a documented descriptive table for a research report.

In [ ]:
import numpy as np
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1004, 1005, 1006],
    "region": ["North", "South", "North", "South", "North", "South"],
    "employed": [True, False, True, True, True, False],
    "wage": [42000., np.nan, 51000., 36000., 62000., np.nan],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print(survey.to_string(index=False))

## Why this matters

Before fitting a regression, researchers describe the analysis sample. A *Table 1* typically reports counts, central tendencies, and dispersion. We use a tiny, fictional wage survey so that every calculation is transparent; these values are not population estimates.

## Count and distribution

Use `describe`, `count`, and `value_counts` to understand valid values. NaN is a missing wage, not a wage of zero.

In [ ]:
print(survey[["wage", "survey_weight"]].describe())
print("Nonmissing wages:", survey["wage"].count())
print(survey["region"].value_counts())

## Build a small Table 1

Group by region, then use named aggregations. Explicit count avoids treating missing wages as observations with zero wage.

In [ ]:
table1 = survey.groupby("region").agg(
    n_people=("person_id", "size"),
    n_wage=("wage", "count"),
    mean_wage=("wage", "mean"),
    sd_wage=("wage", "std"),
).reset_index()
print(table1.to_string(index=False))

## Cross-tabulate two categories

A cross-tabulation counts observations in combinations of categories. It is descriptive, not causal.

In [ ]:
employment_table = pd.crosstab(survey["region"], survey["employed"])
print(employment_table)

## Weighted descriptive mean

Only nonmissing wages should contribute to the weighted average. These fictional weights are for coding practice and do not define a real survey sampling design.

In [ ]:
valid = survey.dropna(subset=["wage"])
weighted_mean = np.average(valid["wage"], weights=valid["survey_weight"])
print("Weighted mean wage:", round(weighted_mean, 2))

## Export tables reproducibly

Exporting a generated table avoids copying numbers by hand. These strings can later be saved into a project `output/tables/` folder.

In [ ]:
table1_csv = table1.to_csv(index=False)
table1_latex = table1.to_latex(index=False, float_format="%.1f")
print(table1_csv.splitlines()[0])
print(table1_latex.splitlines()[0])

## If you know Stata/R

| Task | Python | Stata | R |
| --- | --- | --- | --- |
| Summary | `survey['wage'].describe()` | `summarize wage, detail` | `summary(survey$wage)` |
| Group means | `groupby('region')['wage'].mean()` | `bysort region: summarize wage` | `dplyr::group_by(region) |> summarise(...)` |
| Cross-tabulate | `pd.crosstab(a, b)` | `tabulate a b` | `table(a, b)` |
| CSV table | `table.to_csv(index=False)` | `export delimited` | `write.csv(table, row.names=FALSE)` |

Weighted means are *not* substitutes for a full complex-survey variance estimator.

## Try it

Each learner answer begins blank; the following check prints a hint and does not interrupt an unfinished notebook.

### ★ Exercise 1 — Report nonmissing wages

Set `observed_wages` to the number of nonmissing entries in the wage column.

In [ ]:
observed_wages = int(survey['wage'].count())

In [ ]:
if isinstance(observed_wages, (int, np.integer)) and observed_wages == int(survey["wage"].count()):
    print("✓ Valid observation count")
else:
    print("Hint: use count() to exclude missing wages.")

### ★★ Exercise 2 — Compute the South-region mean

Assign `south_mean` to the mean of nonmissing wages among South respondents.

In [ ]:
south_mean = survey.loc[survey["region"] == "South", "wage"].mean()

In [ ]:
expected = survey.loc[survey["region"] == "South", "wage"].mean()
if isinstance(south_mean, (int, float, np.number)) and np.isclose(south_mean, expected):
    print("✓ Group mean correct")
else:
    print("Hint: filter the rows before taking the wage mean.")

### ★★ Exercise 3 — Make an employment table

Create `employment_counts` using pandas crosstab, with region as rows and employment as columns.

In [ ]:
employment_counts = pd.crosstab(survey["region"], survey["employed"])

In [ ]:
expected = pd.crosstab(survey["region"], survey["employed"])
if isinstance(employment_counts, pd.DataFrame) and employment_counts.equals(expected):
    print("✓ Cross-tabulation correct")
else:
    print("Hint: pass the region and employed Series into pd.crosstab.")

### ★★★ Exercise 4 — Calculate a weighted average

Assign `weighted_wage` to the mean wage weighted by survey_weight, after excluding missing wages.

In [ ]:
wage_sample = survey.dropna(subset=["wage"])
weighted_wage = np.average(wage_sample["wage"], weights=wage_sample["survey_weight"])

In [ ]:
expected = np.average(valid["wage"], weights=valid["survey_weight"])
if isinstance(weighted_wage, (int, float, np.number)) and np.isclose(weighted_wage, expected):
    print("✓ Weighted mean correct")
else:
    print("Hint: remove missing wage rows before weighting.")

## Common errors

- `KeyError: 'wages'` — column names are exact; here the column is `wage`.
- An average computed after `fillna(0)` treats unreported wages as zero and answers a different question.
- `ZeroDivisionError` or invalid weighted result can arise when all weights vanish; validate sampling weights first.
- `to_markdown()` can require an extra optional formatting dependency; `to_csv()` works without that dependency.

## Recap

- Count nonmissing observations separately from rows.
- Group and cross-tabulate before modeling to inspect sample composition.
- Use only valid observations for a weighted wage mean.
- Export research tables from code rather than manually typing results.

## Next steps

Next: [Matplotlib basics](02_matplotlib_basics.ipynb). Read the [pandas groupby guide](https://pandas.pydata.org/docs/user_guide/groupby.html).

[Report a lesson content issue](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md) with this notebook's path.